# 04. replaceWhere - 特定の範囲だけを入れ替える

日次バッチでよくある状況を考えます。

「9月11日のデータに間違いがあったので、その日の分だけ作り直したい」

素直にやろうとすると、どれも具合が悪いことに気づきます。

| やり方 | 問題 |
|---|---|
| `INSERT` で追加する | 元のデータが残ったままなので、その日の分が二重になる |
| テーブル全体を上書きする | 他の日のデータまで消える |
| `DELETE` してから `INSERT` する | 2段階になる。途中で落ちるとデータが消えたままになる |

この「特定の範囲だけを、まとめて入れ替える」を1回の操作でやるのが `replaceWhere` です。

このノートブックで確かめること:

1. パーティションとは何か
2. 素朴な方法だと何が起きるか
3. `replaceWhere` で範囲を指定して入れ替える
4. 何度実行しても結果が変わらないこと
5. 条件と書き込むデータが食い違うとどうなるか

**前提**: `00_setup` を実行済みであること。

## 準備

In [ ]:
from datetime import date

from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.profile("free").serverless(True).getOrCreate()

In [ ]:
CATALOG = "tech_survey"
TABLE = f"{CATALOG}.silver.daily_orders"

# 行を作るときに毎回書くので、列の定義をまとめておく
SCHEMA = "order_id INT, product STRING, amount INT, order_date DATE"

## 1. 日付でパーティションを切ったテーブルを作る

**パーティション** とは、ある列の値ごとにファイルの置き場所を分ける仕組みです。
`order_date` でパーティションを切ると、日付ごとに別のフォルダにファイルが置かれます。

こうしておくと「9月11日のデータ」を扱うときに、その日のフォルダだけを見ればよくなります。
日次で処理する設計と相性が良いので、ここでも日付で切ります。

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")

spark.sql(f"""
    CREATE TABLE {TABLE} (
        order_id INT,
        product STRING,
        amount INT,
        order_date DATE
    )
    PARTITIONED BY (order_date)
""")

spark.sql(f"""
    INSERT INTO {TABLE} VALUES
        (1, 'laptop',   150000, DATE '2026-09-10'),
        (2, 'monitor',   40000, DATE '2026-09-10'),
        (3, 'keyboard',  12000, DATE '2026-09-11'),
        (4, 'mouse',      5000, DATE '2026-09-11'),
        (5, 'headset',   20000, DATE '2026-09-12')
""")

display(spark.table(TABLE).orderBy("order_date", "order_id"))

In [ ]:
# 日付ごとの件数。これが後でどう変わるかを見ていく
display(spark.sql(f"SELECT order_date, count(*) AS cnt FROM {TABLE} GROUP BY order_date ORDER BY order_date"))

## 2. 素朴に追加するとどうなるか

9月11日のデータを作り直したい、という状況です。
新しい内容を `INSERT` で入れてみます。

In [ ]:
fixed_rows = spark.createDataFrame(
    [
        (3, "keyboard", 13000, date(2026, 9, 11)),
        (4, "mouse", 5500, date(2026, 9, 11)),
    ],
    SCHEMA,
)

fixed_rows.write.format("delta").mode("append").saveAsTable(TABLE)

display(spark.table(TABLE).where("order_date = DATE '2026-09-11'").orderBy("order_id"))

古い行と新しい行が両方残ってしまいました。
作り直したかっただけなのに、9月11日のデータが二重になっています。

かといってテーブル全体を上書き (`mode("overwrite")`) すると、
触るつもりのなかった9月10日と12日まで消えます。

## 3. `replaceWhere` で入れ替える

まず、さっき二重にしてしまった状態を元に戻します。

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")

spark.sql(f"""
    CREATE TABLE {TABLE} (
        order_id INT,
        product STRING,
        amount INT,
        order_date DATE
    )
    PARTITIONED BY (order_date)
""")

spark.sql(f"""
    INSERT INTO {TABLE} VALUES
        (1, 'laptop',   150000, DATE '2026-09-10'),
        (2, 'monitor',   40000, DATE '2026-09-10'),
        (3, 'keyboard',  12000, DATE '2026-09-11'),
        (4, 'mouse',      5000, DATE '2026-09-11'),
        (5, 'headset',   20000, DATE '2026-09-12')
""")

`mode("overwrite")` に `replaceWhere` を添えると、上書きする範囲を条件で絞れます。

```
.mode("overwrite")                                  上書きする
.option("replaceWhere", "order_date = '2026-09-11'")  ただしこの条件に合う範囲だけ
```

条件に合う既存の行が消え、書き込むデータが入ります。
この2つは**1回の操作としてまとめて行われます**。途中の中途半端な状態が他から見えることはありません。

In [ ]:
fixed_rows = spark.createDataFrame(
    [
        (3, "keyboard", 13000, date(2026, 9, 11)),
        (4, "mouse", 5500, date(2026, 9, 11)),
    ],
    SCHEMA,
)

(
    fixed_rows.write.format("delta")
    .mode("overwrite")
    .option("replaceWhere", "order_date = '2026-09-11'")
    .saveAsTable(TABLE)
)

display(spark.table(TABLE).orderBy("order_date", "order_id"))

## 4. 何度実行しても同じか

日次バッチでは、同じ処理が2回動いてしまうことがあります。
ジョブのリトライ、手動での再実行、上流からの再送などです。

さきほどとまったく同じ書き込みをもう一度実行します。件数がどうなるか予想してください。

In [ ]:
(
    fixed_rows.write.format("delta")
    .mode("overwrite")
    .option("replaceWhere", "order_date = '2026-09-11'")
    .saveAsTable(TABLE)
)

display(spark.sql(f"SELECT order_date, count(*) AS cnt FROM {TABLE} GROUP BY order_date ORDER BY order_date"))

## 5. 条件と書き込むデータが食い違うと

`replaceWhere` には安全装置があります。
「9月11日を入れ替える」と宣言したのに、9月12日のデータが混ざっていたらどうなるでしょうか。

In [ ]:
wrong_rows = spark.createDataFrame(
    [
        (3, "keyboard", 13000, date(2026, 9, 11)),
        (99, "cable", 1000, date(2026, 9, 12)),  # 宣言した範囲の外
    ],
    SCHEMA,
)

try:
    (
        wrong_rows.write.format("delta")
        .mode("overwrite")
        .option("replaceWhere", "order_date = '2026-09-11'")
        .saveAsTable(TABLE)
    )
except Exception as e:
    print(type(e).__name__)
    print(str(e)[:400])

拒否されたはずです。

もしこれが通ってしまうと、「9月11日を入れ替えたつもりが、9月12日に知らないデータが増えていた」
という事故が起きます。宣言した範囲と実際に書くデータが一致していることを、Deltaが確認してくれています。

## 6. パーティション列以外でも使えるか

`replaceWhere` の条件は、パーティション列でなくても書けます。
ただし速度が変わります。

- パーティション列が条件 … 該当するフォルダのファイルを差し替えるだけで済む
- それ以外の列が条件 … 条件に合う行を探すために、広い範囲のファイルを読んで書き直す必要がある

動きはしますが、日次更新のように定期的に回す処理では、
**入れ替えたい単位でパーティションを切っておく** のが基本になります。

In [ ]:
display(spark.sql(f"DESCRIBE DETAIL {TABLE}").select("partitionColumns", "numFiles", "sizeInBytes"))

## 考えてみる

- `DELETE` してから `INSERT` する方法と比べて、`replaceWhere` は何が優れているのでしょうか
- `01` のチェックポイントによる重複防止と、`replaceWhere` による重複防止は何が違うのでしょうか
- 9月11日のデータが「0件になった」ことを表現したい場合、どう書けばよいでしょうか

### 答え

**Q1. `DELETE` + `INSERT` と比べて何が優れているか**

2つあります。

1つ目は **途中で落ちても壊れない** ことです。`DELETE` と `INSERT` は別々の操作なので、
間で失敗すると「その日のデータが消えたまま」の状態が残ります。
`replaceWhere` は1回の操作なので、成功か、何も起きていないかのどちらかにしかなりません。

2つ目は **他から中途半端な状態が見えない** ことです。`DELETE` の直後に誰かがそのテーブルを
読むと、データが欠けた状態を見てしまいます。`replaceWhere` ではその瞬間が存在しません。

**Q2. チェックポイントによる重複防止との違い**

守っている場所が違います。

- チェックポイント … **読む側** が「どこまで読んだか」を覚えている。同じ入力を2度読まない
- `replaceWhere` … **書く側** が「どの範囲を担当するか」を宣言する。同じ範囲を何度書いても結果は1つ

チェックポイントは、上流から同じデータが再送されてきた場合には無力です。読む側にとっては
新しいファイルなので取り込んでしまいます。
一方 `replaceWhere` は、何が来ようとその範囲を宣言した内容で置き換えるので、再送に強くなります。

日次バッチでは「その日の分をもう一度作り直す」ことが頻繁に起きるので、
書く側で範囲を宣言できる `replaceWhere` が向いています。

**Q3. 0件になったことを表現するには**

空のDataFrameを `replaceWhere` で書き込みます。
条件に合う既存の行が消え、入るものが無いので、結果としてその日は0件になります。

```python
empty = spark.createDataFrame([], SCHEMA)
(
    empty.write.format("delta")
    .mode("overwrite")
    .option("replaceWhere", "order_date = '2026-09-11'")
    .saveAsTable(TABLE)
)
```

「データが無いので何もしない」という実装にしてしまうと、古いデータが残り続けます。
上流で取り消しが起きた場合に間違った結果になるので、注意が必要なところです。

## 後片付け

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")